# 03 — Baseline
Bag of Words + Multinomial Naive Bayes. Este es el punto de referencia mínimo contra el cual se comparan todos los demás experimentos.

In [3]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
from dotenv import load_dotenv
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings('ignore')
load_dotenv()

RANDOM_SEED = int(os.getenv('RANDOM_SEED', 42))
MLFLOW_TRACKING_URI    = os.getenv('MLFLOW_TRACKING_URI', 'http://ec2-52-203-38-164.compute-1.amazonaws.com:5000')
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'sentiment140')

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

np.random.seed(RANDOM_SEED)

DATA_ENCODED = Path('../data/encoded')
MODELS_DIR   = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete')

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Setup complete


## Carga de datos

In [4]:
X_train_bow = sp.load_npz(DATA_ENCODED / 'X_train_bow.npz')
X_test_bow  = sp.load_npz(DATA_ENCODED / 'X_test_bow.npz')
y_train     = np.load(DATA_ENCODED / 'y_train.npy')
y_test      = np.load(DATA_ENCODED / 'y_test.npy')

print(f'Train: {X_train_bow.shape} | Test: {X_test_bow.shape}')

Train: (1360000, 50000) | Test: (240000, 50000)


## Entrenamiento — Multinomial Naive Bayes

In [5]:
def _metrics(y_true, y_pred, prefix: str) -> dict:
    """Compute accuracy/precision/recall/f1 and prefix all keys."""
    return {
        f'{prefix}_accuracy':  accuracy_score(y_true, y_pred),
        f'{prefix}_precision': precision_score(y_true, y_pred, average='macro'),
        f'{prefix}_recall':    recall_score(y_true, y_pred, average='macro'),
        f'{prefix}_f1_macro':  f1_score(y_true, y_pred, average='macro'),
    }


params = {'alpha': 1.0}  # Laplace smoothing

with mlflow.start_run(run_name='baseline_bow_mnb') as run:
    mlflow.set_tag('member', 'team')
    mlflow.set_tag('stage', 'baseline')
    mlflow.set_tag('model', 'MultinomialNB')
    mlflow.set_tag('encoding', 'BoW')

    mlflow.log_params({
        'model': 'MultinomialNB',
        'encoding': 'BoW',
        'random_seed': RANDOM_SEED,
        **params,
    })

    clf = MultinomialNB(**params)
    clf.fit(X_train_bow, y_train)

    train_preds = clf.predict(X_train_bow)
    test_preds  = clf.predict(X_test_bow)

    train_metrics = _metrics(y_train, train_preds, 'train')
    test_metrics  = _metrics(y_test,  test_preds,  'test')

    mlflow.log_metrics({**train_metrics, **test_metrics})

    # Log model artifact
    mlflow.sklearn.log_model(clf, 'model')

    # Log model card artifact
    model_card = {
        'model_name': 'MultinomialNB',
        'encoding': 'BoW',
        'parameters': params,
        'member': 'team',
        'stage': 'baseline',
        'task': 'Binary Sentiment Classification',
        'dataset': 'Sentiment140',
        'metrics': {
            'train': {k.replace('train_', ''): round(v, 4) for k, v in train_metrics.items()},
            'test':  {k.replace('test_',  ''): round(v, 4) for k, v in test_metrics.items()},
        },
        'created_at': pd.Timestamp.now().isoformat(),
    }
    mlflow.log_dict(model_card, 'model_card.json')

    run_id = run.info.run_id

# Keep legacy aliases for downstream cells
preds = test_preds
acc  = test_metrics['test_accuracy']
prec = test_metrics['test_precision']
rec  = test_metrics['test_recall']
f1   = test_metrics['test_f1_macro']

print('=== Baseline Results ===')
print(f'{"Split":<8}  {"Accuracy":>9}  {"Precision":>9}  {"Recall":>9}  {"F1 macro":>9}')
print('-' * 55)
for prefix, metrics in [('Train', train_metrics), ('Test', test_metrics)]:
    print(f'{prefix:<8}  '
          f'{metrics[f"{prefix.lower()}_accuracy"]:>9.4f}  '
          f'{metrics[f"{prefix.lower()}_precision"]:>9.4f}  '
          f'{metrics[f"{prefix.lower()}_recall"]:>9.4f}  '
          f'{metrics[f"{prefix.lower()}_f1_macro"]:>9.4f}')
print(f'\nMLflow run_id: {run_id}')

2026/03/09 04:36:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:36:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run baseline_bow_mnb at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/b2c1dd4a7d12439b9d52a528afd596db
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
=== Baseline Results ===
Split      Accuracy  Precision     Recall   F1 macro
-------------------------------------------------------
Train        0.7903     0.7904     0.7903     0.7902
Test         0.7822     0.7823     0.7822     0.7822

MLflow run_id: b2c1dd4a7d12439b9d52a528afd596db


In [6]:
print(classification_report(y_test, preds, target_names=['Negative', 'Positive']))

              precision    recall  f1-score   support

    Negative       0.78      0.79      0.78    120129
    Positive       0.79      0.77      0.78    119871

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000

